# Stage 1: Synthetic Data Generation & Experimental Protocol

Defines the randomised hidden planted-interaction benchmark: seals the private oracle, publishes the truth-free candidate registry, and generates a **null-condition reference panel** for Stage 2 (so preprocessing/EDA cannot leak any hidden interaction). The 15 planted panels (5 seeds x 3 strengths) and 5 structural panels are regenerated on demand by the Stage 3 runner. Borrowers and macro data are the frozen generator, untouched.

In [1]:
# imports and the fixed experiment settings shared with every later stage
import json, numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display
# the simulator
import credit_dgp as dgp
# shared constants and the public registry helper
from pipeline_utils import (public_candidate_registry, save_json,
    REDUCED_SEEDS, REDUCED_STRENGTHS, SPLIT_DEV_END, SPLIT_VAL_END)
# folder for every saved result
OUT=Path('outputs'); OUT.mkdir(exist_ok=True)
N_ORIG=600            # loans originated per quarter (protocol)
PAIRED_SEED_BASE=777  # per-quarter CRN base for the paired strength streams
print('protocol -> seeds', REDUCED_SEEDS, '| strengths', REDUCED_STRENGTHS,
      '| split dev<=', SPLIT_DEV_END, 'val<=', SPLIT_VAL_END)

protocol -> seeds [1, 2, 3, 4, 5] | strengths ['null', 'weak', 'moderate', 'strong'] | split dev<= 2016Q4 val<= 2019Q4


## 1.7/1.8/1.10 Seal the private oracle; publish the truth-free registry

In [2]:
# select the hidden active set (private) and seal it. Stage 3 discovery must not read
# this file until the freeze-then-reveal step. It is not printed here, kept blind.
hidden=dgp.select_hidden_planted_set(seed=20240824, n_active=5)
save_json({'hidden_active_set':hidden, 'strengths':dgp.PLANTED_STRENGTHS,
           'note':'SEALED private planted oracle (active pairs, signs, amplitudes). '
                  'Do NOT open before the Stage 3 freeze.'},
          OUT/'oracle_planted_private.json')
# public, truth-free registry: pair names only, no labels
registry=public_candidate_registry()
save_json({'candidate_pairs':registry,'note':'public 35-pair grid, names only (no truth labels)'},
          OUT/'registry_public.json')
# experiment manifest, saved for reproducibility
manifest={'seeds':REDUCED_SEEDS,'strengths':REDUCED_STRENGTHS,'n_orig_per_quarter':N_ORIG,
          'paired_seed_base':PAIRED_SEED_BASE,'structural_seeds':REDUCED_SEEDS,
          'split':{'dev_end':SPLIT_DEV_END,'val_end':SPLIT_VAL_END},
          'vasicek':False,'macro_lag':False,'hidden_set_seed':20240824,
          'claim_boundary':'methodological recovery in a UK-informed simulator; '
                           'not causal or true UK/Lloyds coefficients'}
save_json(manifest, OUT/'experiment_manifest.json')
print('sealed private oracle, public registry and manifest saved; %d candidate pairs.'%len(registry))

sealed private oracle, public registry and manifest saved; 35 candidate pairs.


## 1.2-1.6 Reference development panel (NULL condition, seed 1)

In [3]:
# null-condition reference panel: no planted interactions, so Stage 2 EDA and preprocessing
# cannot reveal any hidden pair. direct Bernoulli draw (no Vasicek), so the oracle later is exact.
cfg_null=dgp.make_interaction_config(hidden,'null')
ref=dgp.generate_panel(seed=REDUCED_SEEDS[0], n_orig_per_quarter=N_ORIG,
                       interaction_config=cfg_null, use_vasicek=False, paired_seed=PAIRED_SEED_BASE)
# save the panel that Stage 2 reads in
ref[dgp.STORED_MICRO_COLUMNS+dgp.MACRO_COLUMNS].to_csv('synthetic_credit_panel.csv',index=False)
print('reference (null, seed 1) panel:', ref.shape,
      '| next-q default rate', round(ref['default_next_quarter'].mean()*100,3),'%')

reference (null, seed 1) panel: (574874, 50) | next-q default rate 0.916 %


In [4]:
# public ground-truth params (base hazard, main effects, reference moments, emergent params),
# for rebuilding the oracle's main-effect surface later. Does not contain the hidden set.
with open('ground_truth_params.json','w') as f:
    json.dump({**{k:v for k,v in dgp.TRUE_PARAMS.items()},
        'pop_stats':{k:float(v) for k,v in dgp.POP_STATS.items()},
        'hist_stats':{k:{'mu':v[0],'sd':v[1]} for k,v in dgp.HIST_STATS.items()},
        'winsor_sd':dgp.WINSOR,'m_weights':dgp.M_WEIGHTS,'emergent_params':dgp.EMERGENT_PARAMS,
        'n_orig_per_quarter':N_ORIG,'seed':REDUCED_SEEDS[0]},f,indent=2)
print('ground_truth_params.json saved (public params only).')

ground_truth_params.json saved (public params only).


## 1.13 DGP validation

In [5]:
# sanity checks: unique loan-quarters, at most one default per loan, event rate looks right,
# and terminal rows (default, prepaid, closed) carry no forward target
assert ref.groupby('loan_id')['observation_quarter'].apply(lambda s: s.is_unique).all(), 'dup loan-quarter'
assert (ref['default_next_quarter'].fillna(0).groupby(ref['loan_id']).sum()<=1).all(), '>1 default/loan'
term=ref['performance_status_t'].isin(['default','prepaid','closed','90+ DPD'])
assert ref.loc[term,'default_next_quarter'].isna().all(), 'terminal rows must carry no target'
print('DGP validation OK: unique loan-quarters; <=1 default/loan; rate',
      round(ref['default_next_quarter'].mean()*100,3),'%; loans', f"{ref['loan_id'].nunique():,}")

DGP validation OK: unique loan-quarters; <=1 default/loan; rate 0.916 %; loans 57,000
